# Pasar del dataset de partidas a los csvs con la información necesaria

En esta primera celda lo que vamos a hacer es convertir el jsonl a csv, y obtener solo:
- matchId
- championName
- championId
- teamPosition
- teamId
- win

In [1]:
import json
import csv

file_path = "data/matches_dataset.jsonl"
csv_path = "data/matches_dataset.csv"

positions = ["TOP", "JUNGLE", "MIDDLE", "BOTTOM", "UTILITY"]

# solo ponemos blue_win porque o gana o pierde
headers = ["matchId", "blue_win"]
for pos in positions:
    headers.append(f"blue_{pos}")
for pos in positions:
    headers.append(f"red_{pos}")

with open(file_path, "r", encoding="utf-8") as f_in, open(
    csv_path, "w", newline="", encoding="utf-8"
) as f_out:

    writer = csv.DictWriter(f_out, fieldnames=headers)
    writer.writeheader()

    for line in f_in:
        line = line.strip()
        if not line:
            continue

        match_data = json.loads(line)
        info_match = match_data.get("info", {})
        participants = info_match.get("participants", [])

        # menos de 10 jugadores o menos de 15 minutos, partida corrupta
        if len(participants) != 10 or (info_match.get('gameDuration') < 900):
            continue

        row = {
            "matchId": match_data.get("metadata", {}).get("matchId"),
            "blue_win": None
        }

        # Sets para controlar que Riot no nos envíe roles duplicados en un mismo equipo
        blue_roles = set()
        red_roles = set()
        is_valid_match = True

        for p in participants:
            team_prefix = "blue" if p.get("teamId") == 100 else "red"
            role = p.get("teamPosition")
            champ_id = p.get("championId")

            # Solo necesitamos mirar un jugador azul para saber si ganaron
            if team_prefix == "blue" and row["blue_win"] is None:
                row["blue_win"] = 1 if p.get("win") else 0

            # Verificamos que el rol es valido y no está repetido en el equipo
            if role in positions:
                if team_prefix == "blue":
                    if role in blue_roles:
                        is_valid_match = False
                        break
                    blue_roles.add(role)
                else:
                    if role in red_roles:
                        is_valid_match = False
                        break
                    red_roles.add(role)
                
                row[f"{team_prefix}_{role}"] = champ_id
            else:
                is_valid_match = False
                break

        # Solo guardamos la partida si ambos equipos tienen exactamente los 5 roles distintos
        if is_valid_match and len(blue_roles) == 5 and len(red_roles) == 5:
            writer.writerow(row)

En esta segunda celda crearemos medias de los campeones que se han jugado en las 60k partidas para obtener daño ad, daño ap, curación a aliados, cc...

In [2]:
import json
import pandas as pd

accum_stats = {}

with open("data/matches_dataset.jsonl", "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue

        match_data = json.loads(line)
        info_match = match_data.get("info", {})
        participants = info_match.get("participants", [])

        # menos de 10 jugadores o menos de 15 minutos, partida corrupta
        if len(participants) != 10 or (info_match.get('gameDuration') < 900):
            continue

        for p in participants:
            champ_id = p.get("championId")
            role = p.get("teamPosition")

            if not role or role not in ["TOP", "JUNGLE", "MIDDLE", "BOTTOM", "UTILITY"]:
                continue

            key = f"{champ_id}_{role}"

            if key not in accum_stats:
                accum_stats[key] = {
                    "championId": champ_id,
                    "teamPosition": role,
                    "playedGames": 0,
                    "totalGameDuration": 0,
                    
                    # dmg
                    "totalDamageDealtToChampions": 0,
                    "physicalDamageDealtToChampions": 0,
                    "magicDamageDealtToChampions": 0,
                    "trueDamageDealtToChampions": 0,
                    
                    # tank
                    "damageSelfMitigated": 0,
                    "totalDamageTaken": 0,
                    "totalHeal": 0,
                    
                    # Utility
                    "timeCCingOthers": 0,
                    "totalHealsOnTeammates": 0,
                    "totalDamageShieldedOnTeammates": 0,
                    
                    # Macro
                    "damageDealtToObjectives": 0,
                    "visionScore": 0,
                    "goldEarned": 0
                }

            accum_stats[key]["playedGames"] += 1
            accum_stats[key]["totalGameDuration"] += info_match.get("gameDuration", 0)
            accum_stats[key]["totalDamageDealtToChampions"] += p.get("totalDamageDealtToChampions", 0)
            accum_stats[key]["physicalDamageDealtToChampions"] += p.get("physicalDamageDealtToChampions", 0)
            accum_stats[key]["magicDamageDealtToChampions"] += p.get("magicDamageDealtToChampions", 0)
            accum_stats[key]["trueDamageDealtToChampions"] += p.get("trueDamageDealtToChampions", 0)
            
            accum_stats[key]["damageSelfMitigated"] += p.get("damageSelfMitigated", 0)
            accum_stats[key]["totalDamageTaken"] += p.get("totalDamageTaken", 0)
            accum_stats[key]["totalHeal"] += p.get("totalHeal", 0)
            
            accum_stats[key]["timeCCingOthers"] += p.get("timeCCingOthers", 0)
            accum_stats[key]["totalHealsOnTeammates"] += p.get("totalHealsOnTeammates", 0)
            accum_stats[key]["totalDamageShieldedOnTeammates"] += p.get("totalDamageShieldedOnTeammates", 0)
            
            accum_stats[key]["damageDealtToObjectives"] += p.get("damageDealtToObjectives", 0)
            accum_stats[key]["visionScore"] += p.get("visionScore", 0)
            accum_stats[key]["goldEarned"] += p.get("goldEarned", 0)

# convert to dataframe
data_list = list(accum_stats.values())
df = pd.DataFrame(data_list)

df['totalMinutes'] = df['totalGameDuration'] / 60

df['dpm'] = df['totalDamageDealtToChampions'] / df['totalMinutes']
df['perc_physicalDamage'] = df['physicalDamageDealtToChampions'] / df['totalDamageDealtToChampions']
df['perc_magicDamage'] = df['magicDamageDealtToChampions'] / df['totalDamageDealtToChampions']
df['perc_trueDamage'] = df['trueDamageDealtToChampions'] / df['totalDamageDealtToChampions']

df['damageSelfMitigated_pm'] = df['damageSelfMitigated'] / df['totalMinutes']
df['totalDamageTaken_pm'] = df['totalDamageTaken'] / df['totalMinutes']
df['totalHeal_pm'] = df['totalHeal'] / df['totalMinutes']

df['timeCCingOthers_pm'] = df['timeCCingOthers'] / df['totalMinutes']
df['totalHealsOnTeammates_pm'] = df['totalHealsOnTeammates'] / df['totalMinutes']
df['totalDamageShieldedOnTeammates_pm'] = df['totalDamageShieldedOnTeammates'] / df['totalMinutes']

df['damageDealtToObjectives_pm'] = df['damageDealtToObjectives'] / df['totalMinutes']
df['visionScore_pm'] = df['visionScore'] / df['totalMinutes']
df['goldEarned_pm'] = df['goldEarned'] / df['totalMinutes']

df.to_csv("data/champion_empirical_stats.csv", index=False)
        

En esta tercera celda lo que hacemos es pasar el champions.json de data dragon(riot) y el champion_empirical_stats(obtenido en la segunda celda) a un csv conjunto para que el modelo tenga mas información.

In [3]:
import json
import pandas as pd

with open('data/champions.json', 'r', encoding='utf-8') as file:
    data_dragon = json.load(file)

champ_dict = data_dragon['data']

champ_stats_list = []

for champ_name, champ_info in champ_dict.items():
    tags = champ_info.get('tags', [])

    champ_stat = {
        "championId": int(champ_info['key']),
        
        "riot_attack": champ_info['info']['attack'],
        "riot_magic": champ_info["info"]["magic"],
        "riot_difficulty": champ_info["info"]["difficulty"],
        
        "base_hp": champ_info["stats"]["hp"],
        "base_armor": champ_info["stats"]["armor"],
        "base_spellblock": champ_info["stats"]["spellblock"],
        "attackrange": champ_info["stats"]["attackrange"],
        
        "hp_per_level": champ_info["stats"]["hpperlevel"],
        "armor_per_level": champ_info["stats"]["armorperlevel"],
        "ad_per_level": champ_info["stats"]["attackdamageperlevel"]
    }

    # One-Hot Encoding manual para todas los tags posibles
    for tag in ["Fighter", "Tank", "Mage", "Assassin", "Marksman", "Support"]:
        champ_stat[f"tag_{tag}"] = 1 if tag in tags else 0

    champ_stats_list.append(champ_stat)

static_df = pd.DataFrame(champ_stats_list)

empirical_df = pd.read_csv('data/champion_empirical_stats.csv')

stats_df = pd.merge(empirical_df, static_df, on='championId', how='left')

# guardamos el csv final
stats_df.to_csv('data/champion_stats.csv', index=False)